In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/YOLO_Caries_Training.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))

Dataset extracted successfully!
['.config', 'YOLO_Caries_Training', 'drive', 'sample_data']


In [ ]:
import os

dataset_path = "/content/YOLO_Caries_Training"

print("Dataset exists:", os.path.exists(dataset_path))

if os.path.exists(dataset_path):
    print("Folders:", os.listdir(dataset_path))

Dataset exists: True
Folders: ['data.yaml.txt', 'labels', 'images']


In [ ]:
import os

os.rename(
    "/content/YOLO_Caries_Training/data.yaml.txt",
    "/content/YOLO_Caries_Training/data.yaml"
)

print("Successfully renamed to data.yaml")

Successfully renamed to data.yaml


In [ ]:
yaml_path = "/content/YOLO_Caries_Training/data.yaml"

yaml_content = """path: /content/YOLO_Caries_Training

train: images/train
val: images/valid
test: images/test

nc: 2

names:
  0: D
  1: d
"""

with open(yaml_path, "w") as file:
    file.write(yaml_content)

print("data.yaml updated successfully!")
print(open(yaml_path).read())

data.yaml updated successfully!
path: /content/YOLO_Caries_Training

train: images/train
val: images/valid
test: images/test

nc: 2

names:
  0: D
  1: d



In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

print("YOLOv8 installed successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLOv8 installed successfully!


In [ ]:
from pathlib import Path

dataset = Path("/content/YOLO_Caries_Training")

for split in ["train", "valid", "test"]:
    image_dir = dataset / "images" / split
    label_dir = dataset / "labels" / split

    images = {
        p.stem for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    }
    labels = {p.stem for p in label_dir.glob("*.txt")}

    class_ids = set()

    for label_file in label_dir.glob("*.txt"):
        for line in label_file.read_text().splitlines():
            if line.strip():
                class_ids.add(int(line.split()[0]))

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Images without labels:", len(images - labels))
    print("Labels without images:", len(labels - images))
    print("Class IDs found:", sorted(class_ids))


TRAIN
Images: 1781
Labels: 1781
Images without labels: 0
Labels without images: 0
Class IDs found: [0, 1]

VALID
Images: 214
Labels: 214
Images without labels: 0
Labels without images: 0
Class IDs found: [0, 1]

TEST
Images: 232
Labels: 232
Images without labels: 0
Labels without images: 0
Class IDs found: [0, 1]


In [ ]:
from pathlib import Path
import math

dataset = Path("/content/YOLO_Caries_Training")
errors = []
total_boxes = 0

for split in ["train", "valid", "test"]:
    for file in (dataset / "labels" / split).glob("*.txt"):
        for line_number, line in enumerate(
            file.read_text().splitlines(), start=1
        ):
            if not line.strip():
                continue

            parts = line.split()

            try:
                if len(parts) != 5:
                    raise ValueError("Expected 5 values")

                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])

                if class_id not in [0, 1]:
                    raise ValueError("Invalid class ID")

                if not all(math.isfinite(v) for v in [x, y, w, h]):
                    raise ValueError("Non-finite coordinate")

                if not (0 <= x <= 1 and 0 <= y <= 1):
                    raise ValueError("Invalid box center")

                if not (0 < w <= 1 and 0 < h <= 1):
                    raise ValueError("Invalid box size")

                if (x - w/2 < -1e-5 or x + w/2 > 1.00001 or
                    y - h/2 < -1e-5 or y + h/2 > 1.00001):
                    raise ValueError("Bounding box outside image")

                total_boxes += 1

            except ValueError as e:
                errors.append(f"{split}/{file.name}:{line_number} - {e}")

print("Total valid bounding boxes:", total_boxes)
print("Annotation errors:", len(errors))

for error in errors[:10]:
    print(error)

Total valid bounding boxes: 6686
Annotation errors: 42
train/anonymous_003-008-1140-00_1732538036389_Right_Lateral_View.txt:1 - Invalid box center
train/anonymous_003-007-1213-01_1732862150627_Right_Lateral_View.txt:1 - Invalid box center
train/anonymous_003-008-1140-00_1732538036389_Left_Lateral_View.txt:1 - Invalid box center
train/anonymous_003-008-1140-00_1732538036389_Left_Lateral_View.txt:2 - Invalid box center
train/anonymous_003-008-1188-00_1732714947165_Frontal_View.txt:1 - Invalid box center
train/anonymous_003-008-1188-00_1732714947165_Frontal_View.txt:2 - Invalid box center
train/anonymous_003-008-1188-00_1732714947165_Frontal_View.txt:3 - Invalid box center
train/anonymous_003-008-1188-00_1732714947165_Frontal_View.txt:4 - Invalid box center
train/anonymous_003-008-1188-00_1732714947165_Frontal_View.txt:5 - Invalid box center
train/anonymous_003-008-1187-01_1732714576631_Left_Lateral_View.txt:1 - Invalid box center


In [ ]:
from pathlib import Path

label_dir = Path("/content/YOLO_Caries_Training/labels/train")

file = next(
    label_dir.glob("*1732538036389_Right_Lateral_View*.txt")
)

print("Filename:", file.name)
print("Annotation contents:")
print(file.read_text())

Filename: anonymous_003-008-1140-00_1732538036389_Right_Lateral_View.txt
Annotation contents:
0 1.104001322751323 0.3646795090998899 0.1909455538487798 0.32676013526349773



In [ ]:
from PIL import Image
from pathlib import Path

image_dir = Path("/content/YOLO_Caries_Training/images/train")

image_file = next(
    image_dir.glob("*1732538036389_Right_Lateral_View*")
)

with Image.open(image_file) as img:
    print("Image:", image_file.name)
    print("Width:", img.width)
    print("Height:", img.height)

In [ ]:
from pathlib import Path

image_dir = Path("/content/YOLO_Caries_Training/images/train")

matches = [
    p.name for p in image_dir.iterdir()
    if "1732538036389" in p.name
]

print("Matching images:", matches)

In [ ]:
import os

print("Files in Colab:")
print(os.listdir("/content/"))

print("Google Drive connected:",
      os.path.exists("/content/drive/MyDrive"))

Files in Colab:
['.config', 'sample_data']
Google Drive connected: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/YOLO_Caries_Training.zip")
extract_path = Path("/content")

if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    dataset = extract_path / "YOLO_Caries_Training"

    # Fix the configuration filename if necessary
    old_yaml = dataset / "data.yaml.txt"
    new_yaml = dataset / "data.yaml"

    if old_yaml.exists():
        old_yaml.rename(new_yaml)

    print("Dataset restored successfully!")
    print("Dataset folders:", [p.name for p in dataset.iterdir()])
else:
    print("ZIP file not found in Google Drive!")

Dataset restored successfully!
Dataset folders: ['data.yaml', 'labels', 'images']


In [ ]:
from PIL import Image
from pathlib import Path

image_dir = Path("/content/YOLO_Caries_Training/images/train")

matches = list(image_dir.glob("*1732538036389_Right_Lateral_View*"))

if matches:
    image_file = matches[0]

    with Image.open(image_file) as img:
        print("Filename:", image_file.name)
        print("Width:", img.width)
        print("Height:", img.height)
else:
    print("Image not found")

Filename: anonymous_003-008-1140-00_1732538036389_Right_Lateral_View.jpg
Width: 1008
Height: 566


In [ ]:
from PIL import Image
from IPython.display import display

image = Image.open(
    "/content/YOLO_Caries_Training/images/train/"
    "anonymous_003-008-1140-00_1732538036389_Right_Lateral_View.jpg"
)

display(image)

In [ ]:
from pathlib import Path
from collections import Counter

dataset = Path("/content/YOLO_Caries_Training")
issues = Counter()
affected_files = set()

for split in ["train", "valid", "test"]:
    for file in (dataset / "labels" / split).glob("*.txt"):
        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()
            if len(parts) != 5:
                continue

            try:
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                continue

            problem = None

            if x < 0 or x > 1 or y < 0 or y > 1:
                problem = "Center outside image"
            elif w <= 0 or h <= 0 or w > 1 or h > 1:
                problem = "Invalid width or height"
            elif x-w/2 < 0 or x+w/2 > 1 or y-h/2 < 0 or y+h/2 > 1:
                problem = "Box partially outside image"

            if problem:
                issues[problem] += 1
                affected_files.add((split, file.name))

print("Error types:", dict(issues))
print("Total affected files:", len(affected_files))

Error types: {'Center outside image': 42}
Total affected files: 16


In [ ]:
from collections import Counter

split_counts = Counter(split for split, name in affected_files)

print("Affected files by dataset:")
for split in ["train", "valid", "test"]:
    print(f"{split}: {split_counts[split]}")

Affected files by dataset:
train: 12
valid: 2
test: 2


In [ ]:
import shutil
from pathlib import Path

dataset = Path("/content/YOLO_Caries_Training")

source = dataset / "labels"
backup = Path("/content/YOLO_Original_Labels_Backup")

if not backup.exists():
    shutil.copytree(source, backup)
    print("Original labels backed up successfully!")
else:
    print("Backup already exists. No files changed.")

Original labels backed up successfully!


In [ ]:
from pathlib import Path
from collections import Counter

dataset = Path("/content/YOLO_Caries_Training")
results = Counter()

for split in ["train", "valid", "test"]:
    label_dir = dataset / "labels" / split

    for file in label_dir.glob("*.txt"):
        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) != 5:
                continue

            try:
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1):
                left = x - w / 2
                right = x + w / 2
                top = y - h / 2
                bottom = y + h / 2

                if right <= 0 or left >= 1 or bottom <= 0 or top >= 1:
                    results["Completely outside"] += 1
                else:
                    results["Partially inside"] += 1

print("Completely outside:", results["Completely outside"])
print("Partially inside:", results["Partially inside"])
print("Total:", sum(results.values()))

Completely outside: 41
Partially inside: 1
Total: 42


In [ ]:
from pathlib import Path

dataset = Path("/content/YOLO_Caries_Training")

for split in ["train", "valid", "test"]:
    for file in (dataset / "labels" / split).glob("*.txt"):
        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) != 5:
                continue

            try:
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1):
                left = x - w / 2
                right = x + w / 2
                top = y - h / 2
                bottom = y + h / 2

                if right > 0 and left < 1 and bottom > 0 and top < 1:
                    print("Dataset:", split)
                    print("Filename:", file.name)
                    print("Annotation:", line)
                    print("Box edges:", left, top, right, bottom)

Dataset: train
Filename: anonymous_003-008-1188-00_1732714947165_Frontal_View.txt
Annotation: 0 1.058151174556889 0.9676619325337622 0.17300096837020035 0.23444614520160423
Box edges: 0.9716506903717889 0.8504388599329601 1.1446516587419893 1.0848850051345642


In [ ]:
from pathlib import Path
from collections import Counter

dataset = Path("/content/YOLO_Caries_Training")
summary = Counter()

for split, filename in affected_files:
    file = dataset / "labels" / split / filename

    valid = 0
    invalid = 0

    for line in file.read_text().splitlines():
        if not line.strip():
            continue

        parts = line.split()
        x, y, w, h = map(float, parts[1:])

        if (
            0 <= x - w/2 and x + w/2 <= 1
            and 0 <= y - h/2 and y + h/2 <= 1
        ):
            valid += 1
        else:
            invalid += 1

    if valid > 0:
        summary["Files with valid and invalid boxes"] += 1
    else:
        summary["Files with only invalid boxes"] += 1

print(dict(summary))

{'Files with only invalid boxes': 16}


In [ ]:
from pathlib import Path
import shutil

dataset = Path("/content/YOLO_Caries_Training")
excluded = Path("/content/YOLO_Excluded_Images")

removed = {"train": 0, "valid": 0, "test": 0}

for split, filename in affected_files:
    label_file = dataset / "labels" / split / filename

    image_dir = dataset / "images" / split
    matching_images = [
        p for p in image_dir.iterdir()
        if p.stem == label_file.stem
    ]

    if not label_file.exists() or len(matching_images) != 1:
        print("Skipped (missing or ambiguous pair):", filename)
        continue

    backup_labels = excluded / "labels" / split
    backup_images = excluded / "images" / split

    backup_labels.mkdir(parents=True, exist_ok=True)
    backup_images.mkdir(parents=True, exist_ok=True)

    shutil.move(str(label_file), str(backup_labels / label_file.name))

    for image in matching_images:
        shutil.move(str(image), str(backup_images / image.name))

    removed[split] += 1

print("Excluded image-label pairs:", removed)

for split in ["train", "valid", "test"]:
    images = list((dataset / "images" / split).iterdir())
    labels = list((dataset / "labels" / split).glob("*.txt"))

    print(f"{split}: {len(images)} images, {len(labels)} labels")

Excluded image-label pairs: {'train': 12, 'valid': 2, 'test': 2}
train: 1769 images, 1769 labels
valid: 212 images, 212 labels
test: 230 images, 230 labels


In [ ]:
from pathlib import Path

dataset = Path("/content/YOLO_Caries_Training")
errors = []
total_boxes = 0

for split in ["train", "valid", "test"]:
    for file in (dataset / "labels" / split).glob("*.txt"):
        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) != 5:
                errors.append(file.name)
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])

                valid = (
                    cls in [0, 1]
                    and 0 <= x - w/2
                    and x + w/2 <= 1
                    and 0 <= y - h/2
                    and y + h/2 <= 1
                    and w > 0
                    and h > 0
                )

                if valid:
                    total_boxes += 1
                else:
                    errors.append(file.name)

            except ValueError:
                errors.append(file.name)

print("Valid bounding boxes:", total_boxes)
print("Remaining annotation errors:", len(errors))

Valid bounding boxes: 6686
Remaining annotation errors: 0


In [ ]:
%pip install -q ultralytics

from ultralytics import YOLO
from pathlib import Path

dataset = Path("/content/YOLO_Caries_Training")
yaml_path = dataset / "data.yaml"

yaml_content = """path: /content/YOLO_Caries_Training

train: images/train
val: images/valid
test: images/test

nc: 2
names:
  0: D
  1: d
"""

yaml_path.write_text(yaml_content)

print("YOLOv8 installed successfully!")
print("Dataset configuration ready!")
print(yaml_path.read_text())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 11.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLOv8 installed successfully!
Dataset configuration ready!
path: /content/YOLO_Caries_Training

train: images/train
val: images/valid
test: images/test

nc: 2
names:
  0: D
  1: d



In [ ]:
import shutil
from pathlib import Path

dataset_path = Path("/content/YOLO_Caries_Training")
backup_path = Path("/content/drive/MyDrive/YOLO_Caries_Cleaned.zip")

if dataset_path.exists():
    shutil.make_archive(
        "/content/drive/MyDrive/YOLO_Caries_Cleaned",
        "zip",
        root_dir="/content",
        base_dir="YOLO_Caries_Training"
    )

    print("Cleaned dataset saved to Google Drive!")
    print("Backup:", backup_path)
else:
    print("Dataset not found. Please check the Colab runtime.")

Cleaned dataset saved to Google Drive!
Backup: /content/drive/MyDrive/YOLO_Caries_Cleaned.zip


In [ ]:
from ultralytics import YOLO
import torch

print("GPU available:", torch.cuda.is_available())

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/YOLO_Caries_Training/data.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/drive/MyDrive/Dental_Caries_Results",
    name="YOLOv8_First_Training",
    exist_ok=True
)

GPU available: True
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO_Caries_Training/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8_First_Training, nbs=64, 

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/Dental_Caries_Results/"
    "YOLOv8_First_Training/weights/best.pt"
)

metrics = model.val(
    data="/content/YOLO_Caries_Training/data.yaml",
    split="test",
    imgsz=640,
    project="/content/drive/MyDrive/Dental_Caries_Results",
    name="YOLOv8_First_Test",
    exist_ok=True
)

print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP@50:", round(metrics.box.map50, 4))
print("mAP@50-95:", round(metrics.box.map, 4))

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 56.9±27.2 MB/s, size: 141.7 KB)
val: Scanning /content/YOLO_Caries_Training/labels/test... 230 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 230/230 561.0it/s 0.4s
val: New cache created: /content/YOLO_Caries_Training/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.7it/s 5.5s
                   all        230        682      0.699       0.84      0.795      0.634
                     D        211        633      0.779      0.924      0.933      0.744
                     d         31         49      0.619      0.755      0.658      0.524
Speed: 2.5ms preprocess, 7.1ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/drive/MyDrive/Dental_Caries_Results/YOLOv8

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
from pathlib import Path
import random

model = YOLO(
    "/content/drive/MyDrive/Dental_Caries_Results/"
    "YOLOv8_First_Training/weights/best.pt"
)

test_folder = Path(
    "/content/YOLO_Caries_Training/images/test"
)

images = [
    p for p in test_folder.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

random.seed(42)
image_path = random.choice(images)

results = model.predict(
    source=str(image_path),
    conf=0.25,
    save=False
)

print("Test image:", image_path.name)
print("Detections:", len(results[0].boxes))

display(Image.fromarray(
    results[0].plot()[..., ::-1]
))


image 1/1 /content/YOLO_Caries_Training/images/test/anonymous-maxillaryView-1726818272842.jpg: 640x640 6 Ds, 12.2ms
Speed: 2.7ms preprocess, 12.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Test image: anonymous-maxillaryView-1726818272842.jpg
Detections: 6


In [ ]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
from IPython.display import display

model = YOLO(
    "/content/drive/MyDrive/Dental_Caries_Results/"
    "YOLOv8_First_Training/weights/best.pt"
)

image_path = Path(
    "/content/YOLO_Caries_Training/images/test/"
    "anonymous-maxillaryView-1726818272842.jpg"
)

label_path = (
    Path("/content/YOLO_Caries_Training/labels/test")
    / (image_path.stem + ".txt")
)

print("Ground-truth annotations:")
print(label_path.read_text())

print("\nModel predictions:")
results = model.predict(str(image_path), conf=0.25, verbose=False)

for box in results[0].boxes:
    print(
        "Class:", model.names[int(box.cls.item())],
        "Confidence:", round(float(box.conf.item()), 3)
    )

Ground-truth annotations:
0 0.8672541216600341 0.44940306992609436 0.13644115974985788 0.08982376350198977
0 0.8848777714610574 0.5090960773166572 0.1671404206935759 0.13871517907902217
0 0.8797612279704378 0.5812961910176236 0.14553723706651506 0.06708357021034679
0 0.14411597498578738 0.4516770892552587 0.1318931210915293 0.08982376350198977
0 0.12364980102330869 0.5147811256395679 0.16600341102899374 0.1273450824332007
0 0.10943718021603184 0.5875497441728255 0.13530415008527572 0.07276861853325753


Model predictions:
Class: D Confidence: 0.801
Class: D Confidence: 0.783
Class: D Confidence: 0.705
Class: D Confidence: 0.691
Class: D Confidence: 0.512
Class: D Confidence: 0.412


In [ ]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
import torch

# Use the predictions from the previous cell
pred_boxes = results[0].boxes.xyxy.cpu()

# Load ground-truth annotations
gt_data = []

with open(label_path, "r") as f:
    for line in f:
        values = list(map(float, line.split()))
        gt_data.append(values[1:5])

gt_boxes = torch.tensor(gt_data, dtype=torch.float32)

# Convert YOLO coordinates to pixel coordinates
h, w = results[0].orig_shape

x, y, bw, bh = gt_boxes.T

gt_xyxy = torch.stack([
    (x - bw / 2) * w,
    (y - bh / 2) * h,
    (x + bw / 2) * w,
    (y + bh / 2) * h
], dim=1)

# Calculate IoU
iou_matrix = box_iou(pred_boxes, gt_xyxy)

print("IoU values for each predicted box:")
for i, row in enumerate(iou_matrix):
    print(f"Prediction {i+1}: Best IoU = {row.max().item():.3f}")

IoU values for each predicted box:
Prediction 1: Best IoU = 0.883
Prediction 2: Best IoU = 0.833
Prediction 3: Best IoU = 0.825
Prediction 4: Best IoU = 0.926
Prediction 5: Best IoU = 0.836
Prediction 6: Best IoU = 0.893


In [ ]:
from scipy.optimize import linear_sum_assignment

# Reuse the IoU matrix from the previous cell
matrix = iou_matrix.cpu().numpy()

# Find unique prediction-to-annotation matches
pred_indices, gt_indices = linear_sum_assignment(-matrix)

print("Unique prediction matches:")

for pred_idx, gt_idx in zip(pred_indices, gt_indices):
    iou = matrix[pred_idx, gt_idx]
    print(
        f"Prediction {pred_idx + 1} → "
        f"Ground truth {gt_idx + 1}: IoU = {iou:.3f}"
    )

print("\nAverage matched IoU:",
      round(matrix[pred_indices, gt_indices].mean(), 3))

print("Matches with IoU >= 0.5:",
      sum(matrix[pred_indices, gt_indices] >= 0.5))

Unique prediction matches:
Prediction 1 → Ground truth 1: IoU = 0.883
Prediction 2 → Ground truth 4: IoU = 0.833
Prediction 3 → Ground truth 3: IoU = 0.825
Prediction 4 → Ground truth 5: IoU = 0.926
Prediction 5 → Ground truth 6: IoU = 0.836
Prediction 6 → Ground truth 2: IoU = 0.893

Average matched IoU: 0.866
Matches with IoU >= 0.5: 6


In [ ]:
from pathlib import Path
import json

output_dir = Path(
    "/content/drive/MyDrive/Dental_Caries_Results"
)
output_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "model": "YOLOv8n",
    "epochs": 20,
    "training_images": 1769,
    "validation_images": 212,
    "test_images": 230,
    "precision": 0.6989,
    "recall": 0.8396,
    "mAP50": 0.7952,
    "mAP50_95": 0.6340,
    "sample_image": "anonymous-maxillaryView-1726818272842.jpg",
    "sample_matched_boxes": 6,
    "sample_average_iou": 0.866
}

file_path = output_dir / "YOLOv8_Baseline_Summary.json"

with open(file_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation summary saved successfully!")
print("File:", file_path)

Evaluation summary saved successfully!
File: /content/drive/MyDrive/Dental_Caries_Results/YOLOv8_Baseline_Summary.json
